In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Could not infer project root from notebook location.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

print(f"Project root: {PROJECT_ROOT}")


Project root: /mnt/c/Research/LLMAgora


In [2]:
from dataclasses import fields

from agora.eval_aggregate import build_experiment_analysis_dataframe
from agora.experiment import ExperimentConfig, SEMANTIC_ANALYSIS_METRICS, SEMANTIC_SIMILARITY_METHOD_COSINE
from agora.semantic_similarity_analyzer import DEFAULT_COSINE_MODEL_NAME, DEFAULT_NLI_MODEL_NAME
from agora.persona_adherence_evaluator import (
    PERSONA_METRIC_FULL_DEBATE_PRIVATE,
    PERSONA_METRIC_FULL_DEBATE_PUBLIC,
    PERSONA_METRIC_PRIVATE_CUMULATIVE,
    PERSONA_METRIC_PRIVATE_PER_TURN,
    PERSONA_METRIC_PUBLIC_CUMULATIVE,
    PERSONA_METRIC_PUBLIC_PER_TURN,
)
from agora.emotion_analyzer import DEFAULT_EMOTION_MODEL

EXPERIMENT_CONFIG_FIELDS = {field.name for field in fields(ExperimentConfig)}
LLM_JUDGE_METRICS = [
    PERSONA_METRIC_PUBLIC_PER_TURN,
    PERSONA_METRIC_PRIVATE_PER_TURN,
]
RECURSIVE_LLM_JUDGE_METRICS = [
    PERSONA_METRIC_PUBLIC_CUMULATIVE,
    PERSONA_METRIC_PRIVATE_CUMULATIVE,
    PERSONA_METRIC_FULL_DEBATE_PUBLIC,
    PERSONA_METRIC_FULL_DEBATE_PRIVATE,
]

MANIFEST_PATH = PROJECT_ROOT / "outputs" / "sweeps_5" / "manifest.json"

postpro = {
    "device": "cuda",
    "enable_semantic_analysis": True,
    "enable_llm_judge": False,
    "enable_recursive_llm_judge": False,
    "enable_nli": True,
    "nli_model_name": DEFAULT_NLI_MODEL_NAME,
    "enable_emotions": False,
    "emotion_model_name": DEFAULT_EMOTION_MODEL,
    "semantic_similarity_method": SEMANTIC_SIMILARITY_METHOD_COSINE,
    "semantic_similarity_model": DEFAULT_COSINE_MODEL_NAME,
    "persona_scoring_model": "anthropic/claude-sonnet-4.6",
    "persona_score_samples": 3,
}

postpro["semantic_analysis_metrics"] = (
    list(SEMANTIC_ANALYSIS_METRICS)
    if postpro["enable_semantic_analysis"]
    else []
)
postpro["semantic_similarity_device"] = postpro["device"]
postpro["persona_analysis_metrics"] = (
    (list(LLM_JUDGE_METRICS) if postpro["enable_llm_judge"] else [])
    + (
        list(RECURSIVE_LLM_JUDGE_METRICS)
        if postpro["enable_recursive_llm_judge"]
        else []
    )
)

analysis_kwargs = {
    key: value for key, value in postpro.items() if key in EXPERIMENT_CONFIG_FIELDS
}
analysis_kwargs["catalog_path"] = str(PROJECT_ROOT / "data" / "scenarios.json")
analysis_kwargs["prompts_path"] = str(PROJECT_ROOT / "data" / "prompts.json")

analysis_kwargs


{'semantic_similarity_method': 'cosine',
 'semantic_similarity_model': 'all-mpnet-base-v2',
 'persona_scoring_model': 'anthropic/claude-sonnet-4.6',
 'persona_score_samples': 3,
 'semantic_analysis_metrics': ['self_consistency',
  'cross_agent_public_alignment',
  'cross_agent_private_alignment'],
 'semantic_similarity_device': 'cuda',
 'persona_analysis_metrics': [],
 'catalog_path': '/mnt/c/Research/LLMAgora/data/scenarios.json',
 'prompts_path': '/mnt/c/Research/LLMAgora/data/prompts.json'}

In [ ]:
aggregate_df = build_experiment_analysis_dataframe(
    MANIFEST_PATH,
    analysis_kwargs=analysis_kwargs,
    include_nli=postpro["enable_nli"],
    nli_model_name=postpro["nli_model_name"],
    include_emotions=postpro["enable_emotions"],
    emotion_model_name=postpro["emotion_model_name"],
    device=postpro["device"],
)

print(f"Rows: {len(aggregate_df)}")
display(
    aggregate_df[
        [
            "experiment_index",
            "model",
            "incentive_type",
            "incentive_direction",
            "scenario_id",
            "repeat_count",
        ]
    ].head()
)


/home/snoroozi/anaconda3/envs/agora_eval/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loading cosine model: all-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cosine model: all-mpnet-base-v2...


...

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading nli model: dleemiller/finecat-nli-l...


Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

Rows: 30


,experiment_index,model,incentive_type,incentive_direction,scenario_id,repeat_count
0,0,openai/gpt-5.4,future,positive,promotion_committee_max_divergence,3
1,1,openai/gpt-5.4,future,positive,ngo_climate_endorsement_max_divergence,3
2,2,openai/gpt-5.4,future,positive,faculty_manuscript_submission_max_divergence,3
3,3,openai/gpt-5.4,historical,positive,promotion_committee_max_divergence,3
4,4,openai/gpt-5.4,historical,positive,ngo_climate_endorsement_max_divergence,3


In [9]:
#save aggregate_df to pickle
import pickle

with open(PROJECT_ROOT / "outputs" / "sweeps_5" / "aggregate_df.pkl", "wb") as f:
    pickle.dump(aggregate_df, f)

In [10]:
aggregate_df

,experiment_index,config_fingerprint,repeat_count,case_ids,model,incentive_direction,incentive_type,scenario_id,cosine-similarity-self-consistency,cosine-similarity-cross-agent-alignment,persona-individual-turn-scores,persona-cumulative-scores,persona-full-debate-scores,survey-public,survey-private,survey-diff-public-minus-private,decision-self-consistency,decision-cross-agent-alignment,decision-self-consistency-all-repeats,decision-cross-agent-alignment-all-repeats,nli-self-consistency,nli-cross-agent-alignment
0,0,985253df9808b62b50e8feac4dbf73693a0fa58da41d10...,3,"[2c1fc40f4478, a7525b8f6864, 767bf3cb4a4e]",openai/gpt-5.4,positive,future,promotion_committee_max_divergence,"{'alpha': {'debate_turn': [1, 2, 3, 4, 5], 'co...","{'public alignment': {'debate_turn': [1, 2, 3,...","{'alpha': {'public': {'debate_turn': [], 'pers...","{'alpha': {'public': {'debate_turn': [], 'pers...","{'alpha': {'public': {'score': None, 'standard...",{'Q1': {'question': 'I agree with the overall ...,{'Q1': {'question': 'I agree with the overall ...,{'Q1': {'question': 'I agree with the overall ...,"{'decision': 'PROMOTE', 'channel_tuple_orderin...","{'decision': 'PROMOTE', 'agent_tuple_ordering'...","{'decision': 'PROMOTE', 'channel_tuple_orderin...","{'decision': 'PROMOTE', 'agent_tuple_ordering'...","{'alpha': {'debate_turns': [1, 2, 3, 4, 5], 'n...","{'public utterances': {'debate_turns': [1, 2, ..."
1,1,1ae2b606b1707052c428e4e3333623f1113023fe19e394...,3,"[9ccacbbb84f0, 94ac1fc37cbd, 6cece50a8b2b]",openai/gpt-5.4,positive,future,ngo_climate_endorsement_max_divergence,"{'alpha': {'debate_turn': [1, 2, 3, 4, 5], 'co...","{'public alignment': {'debate_turn': [1, 2, 3,...","{'alpha': {'public': {'debate_turn': [], 'pers...","{'alpha': {'public': {'debate_turn': [], 'pers...","{'alpha': {'public': {'score': None, 'standard...",{'Q1': {'question': 'I agree with the overall ...,{'Q1': {'question': 'I agree with the overall ...,{'Q1': {'question': 'I agree with the overall ...,"{'decision': 'ENDORSE', 'channel_tuple_orderin...","{'decision': 'ENDORSE', 'agent_tuple_ordering'...","{'decision': 'ENDORSE', 'channel_tuple_orderin...","{'decision': 'ENDORSE', 'agent_tuple_ordering'...","{'alpha': {'debate_turns': [1, 2, 3, 4, 5], 'n...","{'public utterances': {'debate_turns': [1, 2, ..."
2,2,8ea43720c971e0e085f50e079e1615a35df301cc4e2f86...,3,"[86a6c3d0db39, a43f310e76e7, 7c444b5887a5]",openai/gpt-5.4,positive,future,faculty_manuscript_submission_max_divergence,"{'alpha': {'debate_turn': [1, 2, 3, 4, 5], 'co...","{'public alignment': {'debate_turn': [1, 2, 3,...","{'alpha': {'public': {'debate_turn': [], 'pers...","{'alpha': {'public': {'debate_turn': [], 'pers...","{'alpha': {'public': {'score': None, 'standard...",{'Q1': {'question': 'I agree with the overall ...,{'Q1': {'question': 'I agree with the overall ...,{'Q1': {'question': 'I agree with the overall ...,"{'decision': 'SUBMIT NOW', 'channel_tuple_orde...","{'decision': 'SUBMIT NOW', 'agent_tuple_orderi...","{'decision': 'SUBMIT NOW', 'channel_tuple_orde...","{'decision': 'SUBMIT NOW', 'agent_tuple_orderi...","{'alpha': {'debate_turns': [1, 2, 3, 4, 5], 'n...","{'public utterances': {'debate_turns': [1, 2, ..."
3,3,5dfc058dcc07c031589ab68c5b317de33216223b5135e8...,3,"[04168129ac8a, 5899555f799a, a73983283925]",openai/gpt-5.4,positive,historical,promotion_committee_max_divergence,"{'alpha': {'debate_turn': [1, 2, 3, 4, 5], 'co...","{'public alignment': {'debate_turn': [1, 2, 3,...","{'alpha': {'public': {'debate_turn': [], 'pers...","{'alpha': {'public': {'debate_turn': [], 'pers...","{'alpha': {'public': {'score': None, 'standard...",{'Q1': {'question': 'I agree with the overall ...,{'Q1': {'question': 'I agree with the overall ...,{'Q1': {'question': 'I agree with the overall ...,"{'decision': 'PROMOTE', 'channel_tuple_orderin...","{'decision': 'PROMOTE', 'agent_tuple_ordering'...","{'decision': 'PROMOTE', 'channel_tuple_orderin...","{'decision': 'PROMOTE', 'agent_tuple_ordering'...","{'alpha': {

In [4]:
experiment_index = 0
row = aggregate_df.loc[aggregate_df["experiment_index"] == experiment_index].iloc[0]

print("Available aggregate columns:")
for col in aggregate_df.columns:
    print(f"- {col}")

row["cosine-similarity-self-consistency"]


Available aggregate columns:
- experiment_index
- config_fingerprint
- repeat_count
- case_ids
- model
- incentive_direction
- incentive_type
- scenario_id
- cosine-similarity-self-consistency
- cosine-similarity-cross-agent-alignment
- persona-individual-turn-scores
- persona-cumulative-scores
- persona-full-debate-scores
- survey-public
- survey-private
- survey-diff-public-minus-private
- decision-self-consistency
- decision-cross-agent-alignment
- decision-self-consistency-all-repeats
- decision-cross-agent-alignment-all-repeats
- nli-self-consistency
- nli-cross-agent-alignment


{'alpha': {'debate_turn': [1, 2, 3, 4, 5],
  'cosine_similarity': [0.7408656279246012,
   0.730864425500234,
   0.7824549476305643,
   0.7474340200424194,
   0.7312975724538168],
  'standard_error': [0.0626010200516105,
   0.004989613846467988,
   0.0026858341505028526,
   0.014578136179350418,
   0.012989770348348284]},
 'beta': {'debate_turn': [1, 2, 3, 4, 5],
  'cosine_similarity': [0.9083196719487509,
   0.8535688320795695,
   0.8997095425923666,
   0.8921864628791809,
   0.9037618835767111],
  'standard_error': [0.006751935361056599,
   0.015414428337377283,
   0.023590419945303354,
   0.0036414056939295496,
   0.025306438497662728]}}

In [5]:
row["nli-self-consistency"] if "nli-self-consistency" in aggregate_df.columns else None


{'alpha': {'debate_turns': [1, 2, 3, 4, 5],
  'nli_probabilities': [(0.0007923780382649662,
    0.005368590597451354,
    0.993839035431544),
   (0.001219407679551902, 0.015031562419608235, 0.9837490320205688),
   (0.0009309602270756537, 0.00779770085743318, 0.9912713468074799),
   (0.00043973533077708754, 0.004960095917340368, 0.9946001668771108),
   (0.0007911677239462733, 0.009841970012833675, 0.989366869131724)],
  'nli_probabilities_standard_error': [(0.0003506906547013856,
    0.002519366602395214,
    0.002868976397264097),
   (0.0003997868031346376, 0.005848449773806568, 0.006247106109329834),
   (0.00018766909885587675, 0.002066212034094267, 0.002245876182911752),
   (2.861825300513631e-05, 0.00017272042711450868, 0.00017418760863329076),
   (5.727952797415346e-05, 0.001308995226954134, 0.0012874669376309958)],
  'nli_tuple_ordering': ('entailment', 'neutral', 'contradiction')},
 'beta': {'debate_turns': [1, 2, 3, 4, 5],
  'nli_probabilities': [(0.6494522194067637,
    0.34957

In [6]:
row["survey-public"]


{'Q1': {'question': 'I agree with the overall position of the other participant.',
  'question_group': 'default',
  'alpha': {'debate_turn': [1, 2, 3, 4, 5],
   'response_score': [-0.3333333333333333, 1.0, 1.0, 1.0, 1.0],
   'standard_error': [0.2721655269759087, 0.0, 0.0, 0.0, 0.0]},
  'beta': {'debate_turn': [1, 2, 3, 4, 5],
   'response_score': [2.0, 2.0, 2.0, 2.0, 2.0],
   'standard_error': [0.0, 0.0, 0.0, 0.0, 0.0]}},
 'Q10': {'question': 'Promoting this candidate strengthens the organization’s long-term leadership pipeline.',
  'question_group': 'direct',
  'alpha': {'debate_turn': [1, 2, 3, 4, 5],
   'response_score': [1.0, 1.0, 1.0, 1.0, 1.0],
   'standard_error': [0.0, 0.0, 0.0, 0.0, 0.0]},
  'beta': {'debate_turn': [1, 2, 3, 4, 5],
   'response_score': [1.0, 1.0, 1.0, 1.0, 1.0],
   'standard_error': [0.0, 0.0, 0.0, 0.0, 0.0]}},
 'Q11': {'question': 'Leadership support is influencing this promotion decision.',
  'question_group': 'direct',
  'alpha': {'debate_turn': [1, 2, 3, 